# Data Preprocessing

## Objective

The objective of this notebook is to clean and validate the raw music dataset before feature engineering and recommendation modeling.

The preprocessing focuses on:

- Removing completely empty records
- Checking remaining missing values
- Validating duplicate records and track identifiers
- Inspecting the availability and validity of core audio features
- Saving a clean dataset for subsequent feature engineering and recommendation modeling

The raw dataset is kept unchanged. All preprocessing operations are performed on a working copy, and the resulting dataset is saved separately under `data/processed/`.

In [1]:
import numpy as np
import pandas as pd

In [2]:
DATA_PATH = "../data/raw/data.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

In [3]:
df.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo
0,0.0594,1921.0,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,831667.0,0.211,0.0,4BJqT0PrAfrxzMOxytFOIz,0.878000,10.0,0.665,-20.096,1.0,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",4.0,1921,0.0366,80.954
1,0.9630,1921.0,0.732,['Dennis Day'],0.819,180533.0,0.341,0.0,7xPhfUan2yNtyFG0cUWkt8,0.000000,7.0,0.160,-12.441,1.0,Clancy Lowered the Boom,5.0,1921,0.4150,60.936
2,0.0394,1921.0,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,500062.0,0.166,0.0,1o6I8BglA6ylDMrIELygv1,0.913000,3.0,0.101,-14.850,1.0,Gati Bali,5.0,1921,0.0339,110.339
3,0.1650,1921.0,0.967,['Frank Parker'],0.275,210000.0,0.309,0.0,3ftBPsC5vPBKxYSee08FDH,0.000028,5.0,0.381,-9.316,1.0,Danny Boy,3.0,1921,0.0354,100.109
4,0.2530,1921.0,0.957,['Phil Regan'],0.418,166693.0,0.193,0.0,4d6HGyGT8e121BsdKmw9v6,0.000002,3.0,0.229,-10.096,1.0,When Irish Eyes Are Smiling,2.0,1921,0.0380,101.665


In [5]:
df.shape

(170653, 19)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 170653 entries, 0 to 170652
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           157490 non-null  float64
 1   year              157490 non-null  float64
 2   acousticness      157490 non-null  float64
 3   artists           157490 non-null  str    
 4   danceability      157490 non-null  float64
 5   duration_ms       157490 non-null  float64
 6   energy            157490 non-null  float64
 7   explicit          157490 non-null  float64
 8   id                157490 non-null  str    
 9   instrumentalness  157490 non-null  float64
 10  key               157490 non-null  float64
 11  liveness          157490 non-null  float64
 12  loudness          157490 non-null  float64
 13  mode              157490 non-null  float64
 14  name              157490 non-null  str    
 15  popularity        157490 non-null  float64
 16  release_date      157490 non-nu

## Remove Completely Empty Rows

The raw dataset contains 170,653 rows. Exploratory data analysis identified 13,163 rows in which all 19 columns contain missing values.

These rows do not represent music records and therefore provide no useful information for subsequent analysis or recommendation modeling.

Only completely empty rows are removed at this stage. Rows containing partial missing values would require separate investigation rather than automatic removal.

In [7]:
df = df.dropna(how = 'all')

In [8]:
df.shape

(157490, 19)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 157490 entries, 0 to 157489
Data columns (total 19 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   valence           157490 non-null  float64
 1   year              157490 non-null  float64
 2   acousticness      157490 non-null  float64
 3   artists           157490 non-null  str    
 4   danceability      157490 non-null  float64
 5   duration_ms       157490 non-null  float64
 6   energy            157490 non-null  float64
 7   explicit          157490 non-null  float64
 8   id                157490 non-null  str    
 9   instrumentalness  157490 non-null  float64
 10  key               157490 non-null  float64
 11  liveness          157490 non-null  float64
 12  loudness          157490 non-null  float64
 13  mode              157490 non-null  float64
 14  name              157490 non-null  str    
 15  popularity        157490 non-null  float64
 16  release_date      157490 non-nu

In [10]:
df.isna().sum()

valence             0
year                0
acousticness        0
artists             0
danceability        0
duration_ms         0
energy              0
explicit            0
id                  0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
name                0
popularity          0
release_date        0
speechiness         0
tempo               0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(0)

### Result

After removing completely empty rows, the dataset contains 157,490 records and 19 columns.

No valid music records were removed based solely on partial missingness at this stage.

In [12]:
df['id'].duplicated().sum()

np.int64(0)

In [13]:
df['id'].nunique()

157490

In [14]:
duplicate_ids = df[df['id'].duplicated(keep = False)].sort_values(by = 'id')

duplicate_ids.head()

,valence,year,acousticness,artists,danceability,duration_ms,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo


## Core Audio Feature Validation

The following continuous audio characteristics are selected for initial recommendation modeling:

- Acousticness
- Danceability
- Energy
- Instrumentalness
- Liveness
- Loudness
- Speechiness
- Valence
- Tempo

These features describe different characteristics of the musical content and will form the initial basis of the content-based recommendation representation.

Other variables such as `duration_ms`, `key`, `mode`, `explicit`, and `popularity` are retained in the dataset but are not included in this initial audio-feature group because they have different semantic roles.

In [15]:
audio_features = [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "valence",
    "tempo"
]

In [16]:
df[audio_features].isna().sum()

acousticness        0
danceability        0
energy              0
instrumentalness    0
liveness            0
loudness            0
speechiness         0
valence             0
tempo               0
dtype: int64

In [17]:
df[audio_features].describe().T

,count,mean,std,min,25%,50%,75%,max
acousticness,157490.0,0.509352,0.377327,0.0,0.10600,0.532000,0.903000,0.996
danceability,157490.0,0.537658,0.176132,0.0,0.41500,0.548000,0.669000,0.988
energy,157490.0,0.477625,0.267541,0.0,0.25000,0.463000,0.698000,1.000
instrumentalness,157490.0,0.169714,0.315752,0.0,0.00000,0.000227,0.110000,1.000
liveness,157490.0,0.205484,0.173417,0.0,0.09900,0.136000,0.261000,0.999
loudness,157490.0,-11.533284,5.711133,-60.0,-14.71600,-10.648000,-7.225000,3.855
speechiness,157490.0,0.100117,0.166722,0.0,0.03500,0.045100,0.076175,0.970
valence,157490.0,0.528954,0.263286,0.0,0.31700,0.540000,0.748000,1.000
tempo,157490.0,116.712327,30.734610,0.0,93.21825,114.579000,135.354750,243.507


In [21]:
for feature in audio_features:
    print(f"{feature}: {df[feature].min()} -> {df[feature].max()}")

acousticness: 0.0 -> 0.996
danceability: 0.0 -> 0.988
energy: 0.0 -> 1.0
instrumentalness: 0.0 -> 1.0
liveness: 0.0 -> 0.999
loudness: -60.0 -> 3.855
speechiness: 0.0 -> 0.97
valence: 0.0 -> 1.0
tempo: 0.0 -> 243.507


## Preprocessing Summary

The preprocessing pipeline produced a clean dataset for subsequent feature engineering.

### Dataset changes

| Stage | Rows | Columns |
|---|---:|---:|
| Raw dataset | 170,653 | 19 |
| After removing completely empty rows | 157,490 | 19 |
| Final processed dataset | 157,490 | 19 |

### Validation results

- 13,163 completely empty records were removed.
- No missing values remain in the processed dataset.
- No complete-row duplicates remain.
- No duplicate track IDs were found.
- All nine selected core audio features contain valid non-null values.
- The original raw dataset remains unchanged.

The cleaned dataset is saved as:

`data/processed/music_clean.csv`

This dataset will be used in the subsequent feature engineering stage.

In [19]:
df.to_csv("../data/processed/music_clean.csv", index=False)

## Save and Verify Processed Dataset

The cleaned dataset is saved separately from the raw data so that the original dataset remains reproducible and unchanged.

The saved file is reloaded to verify that the processed dataset was written correctly.

In [20]:
processed_df = pd.read_csv("../data/processed/music_clean.csv", low_memory=False)
processed_df.shape

(157490, 19)